<!-- # Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`. -->

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [2]:
ubuntu_path = "/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv"
mac_path = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path=mac_path,
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df_raw = graph_data.historical_flows_df.copy()

/Users/vladislav/Documents/vlzm/GFDRR/gbp/loaders/dataloader_raw.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.stations_capacities_df['capacity'] = 100


In [3]:
# # Entities
# facilities_df = get_facilities_df(raw.stations_df, raw.depots_df)
# resources_df = get_resources_df(raw.trucks_df)
# commodities_categories_df = get_commodities_categories_df()

# # Attributes
# facilities_geo_df = get_facilities_geo_df(raw.stations_df, raw.depots_df)
# facilities_capacities_df = get_facilities_capacities_df(
#     raw.stations_capacities_df, raw.depot_capacities_df
# )
# facilities_capacities_df["capacity"] = facilities_capacities_df["capacity"]*scale_capacity_factor
# # if capacity < 10 then capacity = 10
# facilities_capacities_df["capacity"] = facilities_capacities_df["capacity"].apply(lambda x: max(x, 10))
# resources_capacities_df = get_resources_capacities_df(raw.trucks_capacities_df)
# facilities_costs_df = get_facilities_costs_df(
#     raw.stations_costs_df, raw.depot_costs_df
# )
# resources_rates_df = get_resources_rates_df(raw.trucks_rates_df)
# commodities_categories_rates_df = get_commodities_categories_rates_df(
#     raw.bike_rates_df
# )

# # Time grid
# period_len = period_len
# t0 = raw.trips_df["started_at"].min().floor("h")
# periods_df = get_periods_df(raw.trips_df, t0, period_len)

# # Historical observations: the marginals of the flow log, assembled by
# # the shared ``observe`` bundle so they match the simulated set below.
# historical_flows_df = get_historical_flows_df(raw.trips_df, t0, period_len)
# historical_resources_df = empty_resources_obs_df()

# Additional:
# - Capacities 
	# - source_capacity_per_commodity_cat
	# - source_capacity_total
	# - planned_target_capacity_per_commodity_cat
	# - planned_target_capacity_total
	# - realized_target_capacity_per_commodity_cat
	# - realized_target_capacity_total
# - Inventory
# - Costs


# historical_flows_df.columns
# Index(['flow_id', 'move_id', 'event_id', 'period_id', 'flow_type',
#        'event_type', 'commodity_category', 'source_id', 'planned_target_id',
#        'realized_target_id', 'start_period', 'planned_end_period',
#        'realized_end_period', 'resource_id', 'quantity', 'reason'],
#       dtype='str')

historical_flows_df = historical_flows_df_raw.copy()

# source_capacity_total
historical_flows_df = pd.merge(
	historical_flows_df, graph_data.facilities_capacities_df, left_on="source_id", right_on="facility_id", how="left")
historical_flows_df = historical_flows_df.rename(columns={"capacity": "source_capacity_total"})
historical_flows_df = historical_flows_df.drop(columns=["facility_id"])

# source_capacity_per_commodity_cat
if "commodity_category" in graph_data.facilities_capacities_df.columns: 
	historical_flows_df = pd.merge(
		historical_flows_df, graph_data.facilities_capacities_df, left_on=["source_id", "commodity_category"], right_on=["facility_id", "commodity_category"], how="left")
	historical_flows_df = historical_flows_df.rename(columns={"capacity": "source_capacity_per_commodity_cat"})
	historical_flows_df = historical_flows_df.drop(columns=["facility_id"])
else:
	historical_flows_df["source_capacity_per_commodity_cat"] = historical_flows_df["source_capacity_total"]


# planned_target_capacity_total
historical_flows_df = pd.merge(
	historical_flows_df, graph_data.facilities_capacities_df, left_on="planned_target_id", right_on="facility_id", how="left")
historical_flows_df = historical_flows_df.rename(columns={"capacity": "planned_target_capacity_total"})
historical_flows_df = historical_flows_df.drop(columns=["facility_id"])

# planned_target_capacity_per_commodity_cat
if "commodity_category" in graph_data.facilities_capacities_df.columns:
	historical_flows_df = pd.merge(
		historical_flows_df, graph_data.facilities_capacities_df, left_on=["planned_target_id", "commodity_category"], right_on=["facility_id", "commodity_category"], how="left")
	historical_flows_df = historical_flows_df.rename(columns={"capacity": "planned_target_capacity_per_commodity_cat"})
	historical_flows_df = historical_flows_df.drop(columns=["facility_id"])
else:
	historical_flows_df["planned_target_capacity_per_commodity_cat"] = historical_flows_df["planned_target_capacity_total"]

# realized_target_capacity_total
historical_flows_df = pd.merge(
	historical_flows_df, graph_data.facilities_capacities_df, left_on="realized_target_id", right_on="facility_id", how="left")
historical_flows_df = historical_flows_df.rename(columns={"capacity": "realized_target_capacity_total"})
historical_flows_df = historical_flows_df.drop(columns=["facility_id"])

# realized_target_capacity_per_commodity_cat
if "commodity_category" in graph_data.facilities_capacities_df.columns:
	historical_flows_df = pd.merge(
		historical_flows_df, graph_data.facilities_capacities_df, left_on=["realized_target_id", "commodity_category"], right_on=["facility_id", "commodity_category"], how="left")
	historical_flows_df = historical_flows_df.rename(columns={"capacity": "realized_target_capacity_per_commodity_cat"})
	historical_flows_df = historical_flows_df.drop(columns=["facility_id"])
else:
	historical_flows_df["realized_target_capacity_per_commodity_cat"] = historical_flows_df["realized_target_capacity_total"]

In [ ]:
# so, facilities_costs_df is not about events. thus it shluldn't be in events journal (historical_flows_df)
# thus i need to take resources_rates_df and commodities_categories_rates_df and merge them into historical_flows_df as transportation rates.
# then i need to merge distances and durations into historical_flows_df
# then i can calculate transportation costs as transportation_cost =  duration * transportation_rate

In [6]:
graph_data.facilities_costs_df

,facility_id,fixed_cost
0,6602.05,0.00
1,5311.08,0.00
2,6789.08,0.00
3,6605.08,0.00
4,5584.04,0.00
...,...,...
2255,depot_6,169.37
2256,depot_7,196.10
2257,depot_8,119.10
2258,depot_9,124.46


In [5]:
graph_data.facilities_df

,facility_id,facility_category
0,6602.05,station
1,5311.08,station
2,6789.08,station
3,6605.08,station
4,5584.04,station
...,...,...
2255,depot_6,depot
2256,depot_7,depot
2257,depot_8,depot
2258,depot_9,depot


In [33]:
graph_data.resources_rates_df.head()

,resource_id,rate
0,truck_1,50.0
1,truck_2,50.0
2,truck_3,50.0
3,truck_4,50.0
4,truck_5,50.0


In [35]:
graph_data.commodities_categories_rates_df.head()

,commodity_category,rate
0,electric_bike,5
1,classic_bike,3


In [31]:
historical_flows_df

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_end_period,resource_id,quantity,reason,source_capacity_total,source_capacity_per_commodity_cat,planned_target_capacity_total,planned_target_capacity_per_commodity_cat,realized_target_capacity_total,realized_target_capacity_per_commodity_cat
0,hist_244666,0,0,0,user_trip,departed,classic_bike,5210.01,5411.08,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
1,hist_198173,0,0,3,user_trip,departed,classic_bike,6575.03,6659.01,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
2,hist_277917,0,0,3,user_trip,departed,classic_bike,7484.05,6659.01,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
3,hist_177716,0,0,6,user_trip,departed,classic_bike,4550.05,4830.02,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
4,hist_253909,0,0,7,user_trip,departed,classic_bike,4196.05,4425.02,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1988285,hist_993009,0,0,686,user_trip,departed,electric_bike,5238.05,5288.09,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
1988286,hist_993009,0,1,686,user_trip,arrived,electric_bike,5238.05,5288.09,5288.09,...,686,<NA>,1,<NA>,1000,1000,1000,1000,1000.0,1000.0
1988287,hist_993623,0,0,686,user_trip,departed,electric_bike,4386.05,4611.03,<NA>,...,<NA>,<NA>,1,<NA>,1000,1000,1000,1000,NaN,NaN
1988288,hist_993623,0,1,686,user_trip,arrived,electric_bike,4386.05,4611.03,4611.03,...,686,<NA>,1,<NA>,1000,1000,1000,1000,1000.0,1000.0


In [3]:
# import numpy as np

# flows = graph_data.historical_flows_df  # .copy() не нужен — исходный фрейм ты не мутируешь

# departed = (
#     flows.loc[flows["event_type"] == "departed",
#               ["period_id", "source_id", "commodity_category", "quantity"]]
#     .groupby(["period_id", "source_id", "commodity_category"], as_index=False)["quantity"].sum()
#     .rename(columns={"source_id": "facility_id"})
# )
# departed["quantity"] *= -1
# departed["event_type"] = "departed"

# arrived = (
#     flows.loc[flows["event_type"] == "arrived",
#               ["period_id", "realized_target_id", "commodity_category", "quantity"]]
#     .groupby(["period_id", "realized_target_id", "commodity_category"], as_index=False)["quantity"].sum()
#     .rename(columns={"realized_target_id": "facility_id"})
# )
# arrived["event_type"] = "arrived"

# total = pd.concat([departed, arrived], ignore_index=True)

# facilities  = np.sort(graph_data.facilities_df["facility_id"].unique())
# commodities = np.sort(graph_data.commodities_categories_df["commodity_category"].unique())
# periods     = np.sort(graph_data.periods_df["period_id"].unique())
# event_types = ["departed", "arrived"]  # departed раньше arrived

# full_index = pd.MultiIndex.from_product(
#     [facilities, commodities, periods, event_types],
#     names=["facility_id", "commodity_category", "period_id", "event_type"],
# )

# total_flows_final = (
#     total.set_index(["facility_id", "commodity_category", "period_id", "event_type"])["quantity"]
#          .reindex(full_index, fill_value=0)
#          .reset_index()
# )

# keys = ["facility_id", "commodity_category"]
# total_flows_final["inventory"] = total_flows_final.groupby(keys, sort=False)["quantity"].cumsum()
# total_flows_final["offset"] = (
#     -total_flows_final.groupby(keys, sort=False)["inventory"].transform("min")
# ).clip(lower=0)
# total_flows_final["inventory_feasible"] = total_flows_final["inventory"] + total_flows_final["offset"]
# total_flows_final["capacity"] = (
#     total_flows_final.groupby(keys, sort=False)["inventory_feasible"].transform("max")
# )

In [5]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, 
                      seed=42, 
                      scenario_id="historical_replay", 
                      demand_scale_factor=1.0,
                      number_of_periods = 30),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df


def _sorted(df):
    return (df.sort_values(["period_id", "facility_id", "commodity_category"])
            .reset_index(drop=True))


# The base scenario reproduces historical demand exactly: the departures marginal
# of the simulated journal equals the historical one. (The per-trip journal is not
# identical, because targets/durations are drawn from the aggregate OD model.)
# pd.testing.assert_frame_equal(_sorted(simulated_departures_df), _sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))
# print("simulated_departures_df == historical_departures_df:",
#       _sorted(simulated_departures_df).equals(_sorted(_sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))))

In [6]:
simulated_flows_df

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,0,17,<NA>,<NA>,1,<NA>
1,sim_4_0,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
2,sim_4_1,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
3,sim_4_2,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,4,29,<NA>,<NA>,1,<NA>
4,sim_5_0,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30217,sim_29_998,0,0,29,user_trip,departed,electric_bike,6025.08,6932.14,<NA>,29,29,<NA>,<NA>,1,<NA>
30218,sim_29_998,0,1,29,user_trip,arrived,electric_bike,6025.08,6932.14,6932.14,29,29,29,<NA>,1,<NA>
30219,sim_29_999,0,0,29,user_trip,departed,classic_bike,6030.04,6115.09,<NA>,29,29,<NA>,<NA>,1,<NA>
30220,sim_29_999,0,1,29,user_trip,arrived,classic_bike,6030.04,6115.09,6115.09,29,29,29,<NA>,1,<NA>
